# Extração de CDI e IBOV no Yahoo Finance

Este notebook mostra como:

- identificar no Yahoo Finance o ticker disponível para o CDI brasileiro
- baixar o histórico do IBOV
- consolidar as séries em um único `DataFrame`
- exportar os dados para CSV

Observação: o ticker do IBOV no Yahoo Finance é `^BVSP`. Já o CDI pode variar de disponibilidade no Yahoo, então o notebook inclui uma etapa de busca/validação antes do download final.

In [ ]:
# Se necessario, descomente a linha abaixo para instalar a biblioteca.
# %pip install yfinance

In [1]:
from pathlib import Path

import pandas as pd
import yfinance as yf

In [2]:
DATA_INICIO = "2015-01-01"
DATA_FIM = None

IBOV_TICKER = "^BVSP"
CDI_TICKER = None  # Preencha apos validar na busca abaixo.

SAIDA_DIR = Path("../data")
SAIDA_DIR.mkdir(parents=True, exist_ok=True)

## 1. Buscar candidatos para o CDI no Yahoo Finance

Como o Yahoo Finance nem sempre expõe o CDI com um ticker óbvio, esta célula tenta localizar candidatos por texto. Se encontrar um ticker adequado, copie o símbolo para `CDI_TICKER` na célula de parâmetros.

In [3]:
consultas = [
    "CDI Brasil",
    "Taxa DI Brasil",
    "Brazil interbank rate",
    "Brazil CDI rate",
]

resultados = []

for consulta in consultas:
    try:
        busca = yf.Search(consulta)
        for item in busca.quotes:
            resultados.append(
                {
                    "consulta": consulta,
                    "symbol": item.get("symbol"),
                    "shortname": item.get("shortname"),
                    "longname": item.get("longname"),
                    "exchange": item.get("exchange"),
                    "quoteType": item.get("quoteType"),
                }
            )
    except Exception as exc:
        print(f"Falha na consulta '{consulta}': {exc}")

candidatos_cdi = (
    pd.DataFrame(resultados)
    .drop_duplicates(subset=["symbol"])
    .sort_values(["consulta", "symbol"], na_position="last")
    .reset_index(drop=True)
)

candidatos_cdi

KeyError: 'consulta'

## 2. Funções auxiliares para download

In [5]:
def baixar_historico(ticker, inicio=DATA_INICIO, fim=DATA_FIM, intervalo="1d"):
    df = yf.download(
        ticker,
        start=inicio,
        end=fim,
        interval=intervalo,
        auto_adjust=False,
        progress=False,
        threads=False,
    )

    if df.empty:
        raise ValueError(f"Nenhum dado retornado para o ticker: {ticker}")

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df = df.reset_index().rename(columns={"Date": "data"})
    df["ticker"] = ticker
    return df


def resumir_serie(df, nome_serie):
    colunas_base = [c for c in ["data", "Open", "High", "Low", "Close", "Adj Close", "Volume", "ticker"] if c in df.columns]
    resumo = df[colunas_base].copy()
    resumo = resumo.rename(columns=lambda c: f"{nome_serie.lower()}_{c.lower().replace(' ', '_')}" if c not in {"data"} else "data")
    return resumo

## 3. Baixar histórico do IBOV

In [8]:
ibov = baixar_historico(IBOV_TICKER)
ibov.tail()

Price,data,Adj Close,Close,High,Low,Open,Volume,ticker
2780,2026-03-18,179640.00000,179640.00000,181551.000000,179576.00000,180409.0000,9750200,^BVSP
2781,2026-03-19,180271.00000,180271.00000,181251.000000,176296.00000,179624.0000,12560400,^BVSP
2782,2026-03-20,176219.00000,176219.00000,180305.000000,175039.00000,180262.0000,15348900,^BVSP
2783,2026-03-23,181932.00000,181932.00000,182973.000000,176221.00000,176221.0000,9874700,^BVSP
2784,2026-03-24,181114.15625,181114.15625,182564.046875,179914.53125,181931.9375,0,^BVSP


## 4. Baixar histórico do CDI

Depois de preencher `CDI_TICKER`, rode a célula abaixo.

In [ ]:
if not CDI_TICKER:
    raise ValueError("Defina CDI_TICKER na célula de parâmetros antes de baixar a série do CDI.")

cdi = baixar_historico(CDI_TICKER)
cdi.head()

## 5. Consolidar IBOV e CDI em um único DataFrame

In [ ]:
ibov_resumo = resumir_serie(ibov, "IBOV")
cdi_resumo = resumir_serie(cdi, "CDI")

base_consolidada = ibov_resumo.merge(cdi_resumo, on="data", how="outer").sort_values("data").reset_index(drop=True)
base_consolidada.head()

## 6. Visualização rápida

In [ ]:
colunas_plot = [col for col in ["ibov_close", "cdi_close"] if col in base_consolidada.columns]
base_consolidada.set_index("data")[colunas_plot].plot(figsize=(14, 6), title="IBOV e CDI - Yahoo Finance");

## 7. Exportar para CSV

In [ ]:
arquivo_saida = SAIDA_DIR / "cdi_ibov_yahoo_finance.csv"
base_consolidada.to_csv(arquivo_saida, index=False)
arquivo_saida